In [1]:
import numpy as np
import pandas as pd

RANK_CHARS = '23456789TJQKA'
SUIT_CHARS = 'cdhs'
def card_str(c): return RANK_CHARS[int(c)>>2] + SUIT_CHARS[int(c)&3]
def board_str(row): return ' '.join(card_str(row[i]) for i in range(4))

## Raw Data (Golden Source)

In [2]:
meta = np.load('../solver_output/meta.npy')
ranges = np.load('../solver_output/ranges.npy')
values = np.load('../solver_output/values.npy')

reach_oop = ranges[:, :1326]
reach_ip  = ranges[:, 1326:]
cfv_oop   = values[:, :1326]
cfv_ip    = values[:, 1326:]

oop_total = reach_oop.sum(axis=1)
ip_total  = reach_ip.sum(axis=1)

df_raw = pd.DataFrame({
    'board': [board_str(meta[i]) for i in range(len(meta))],
    'pot': meta[:, 4].astype(int),
    'stack': meta[:, 5].astype(int),
    'spr': np.round(meta[:, 5] / meta[:, 4], 1),
    'oop_combos': (reach_oop > 0).sum(axis=1),
    'ip_combos': (reach_ip > 0).sum(axis=1),
    'oop_mean_cfv': np.round(np.where(oop_total > 0, (cfv_oop * reach_oop).sum(axis=1) / oop_total, 0), 4),
    'ip_mean_cfv': np.round(np.where(ip_total > 0, (cfv_ip * reach_ip).sum(axis=1) / ip_total, 0), 4),
})

print(f'Raw: meta{meta.shape}  ranges{ranges.shape}  values{values.shape}')
print(f'Storage: {sum(x.nbytes for x in [meta,ranges,values])/1e6:.1f} MB')
df_raw

Raw: meta(100, 6)  ranges(100, 2652)  values(100, 2652)
Storage: 2.1 MB


,board,pot,stack,spr,oop_combos,ip_combos,oop_mean_cfv,ip_mean_cfv
0,2s 5d 6c Td,3200,3500,1.1,1128,1128,-0.4816,0.4816
1,7d Js Qh 9c,2800,2400,0.9,1128,1128,0.3213,-0.3213
2,8c 8s Qs 8d,5400,4300,0.8,1128,1128,0.0753,-0.0753
3,4s Jd Ah Js,3600,9700,2.7,1128,1128,0.3980,-0.3980
4,5h 5s 8c 6d,5000,8800,1.8,1128,1128,-0.0343,0.0343
...,...,...,...,...,...,...,...,...
95,7c Jc Ah Jd,5900,4200,0.7,1128,1128,0.4480,-0.4480
96,3c 3h Qh 4c,4900,9600,2.0,1128,1128,-0.2258,0.2258
97,Th Jc Qd 3h,3500,3600,1.0,1128,1128,-0.3617,0.3617
98,7c 8c Ks 2c,500,1100,2.2,1128,1128,-0.1649,0.1649


## Bucketed Data (projected from raw)

In [3]:
inputs  = np.load('../training_data/inputs.npy')
targets = np.load('../training_data/targets.npy')

K = 1000
board_feat     = inputs[:, :15]
range_oop_buck = inputs[:, 15:15+K]
range_ip_buck  = inputs[:, 15+K:]
cfv_oop_buck   = targets[:, :K]
cfv_ip_buck    = targets[:, K:]

feat_names = ['r1','r2','r3','r4','suit_c','suit_d','suit_h','suit_s',
              'paired','trips','mono','connect','log_spr','pot_frac','stk_frac']

df_buck = pd.DataFrame({
    'board': [board_str(meta[i]) for i in range(len(meta))],
    'oop_active_buckets': (range_oop_buck > 0).sum(axis=1),
    'ip_active_buckets': (range_ip_buck > 0).sum(axis=1),
    'oop_mean_cfv_buck': np.round(cfv_oop_buck.mean(axis=1), 4),
    'ip_mean_cfv_buck': np.round(cfv_ip_buck.mean(axis=1), 4),
})

print(f'Bucketed: inputs{inputs.shape}  targets{targets.shape}')
print(f'Storage: {sum(x.nbytes for x in [inputs,targets])/1e6:.1f} MB')
df_buck

Bucketed: inputs(100, 2015)  targets(100, 2000)
Storage: 1.6 MB


,board,oop_active_buckets,ip_active_buckets,oop_mean_cfv_buck,ip_mean_cfv_buck
0,2s 5d 6c Td,809,809,-0.0237,0.3830
1,7d Js Qh 9c,169,169,0.0299,-0.0446
2,8c 8s Qs 8d,705,705,0.3899,0.3024
3,4s Jd Ah Js,754,754,0.2402,0.0546
4,5h 5s 8c 6d,169,169,0.0428,0.0444
...,...,...,...,...,...
95,7c Jc Ah Jd,754,754,0.3229,-0.2799
96,3c 3h Qh 4c,656,656,0.1034,0.2003
97,Th Jc Qd 3h,754,754,-0.0562,0.1749
98,7c 8c Ks 2c,686,686,-0.0561,0.1039


## Board Features (15-dim)

In [4]:
df_feat = pd.DataFrame(np.round(board_feat, 3), columns=feat_names)
df_feat.insert(0, 'board', [board_str(meta[i]) for i in range(len(meta))])
df_feat

,board,r1,r2,r3,r4,suit_c,suit_d,suit_h,suit_s,paired,trips,mono,connect,log_spr,pot_frac,stk_frac
0,2s 5d 6c Td,0.667,0.333,0.250,0.000,0.25,0.50,0.00,0.25,0.0,0.0,0.0,0.333,0.239,0.478,0.522
1,7d Js Qh 9c,0.833,0.750,0.583,0.417,0.25,0.25,0.25,0.25,0.0,0.0,0.0,0.583,0.200,0.538,0.462
2,8c 8s Qs 8d,0.833,0.500,0.500,0.500,0.25,0.25,0.00,0.50,1.0,1.0,0.0,0.667,0.189,0.557,0.443
3,4s Jd Ah Js,1.000,0.750,0.750,0.167,0.00,0.25,0.25,0.50,1.0,0.0,0.0,0.167,0.423,0.271,0.729
4,5h 5s 8c 6d,0.500,0.333,0.250,0.250,0.25,0.25,0.25,0.25,1.0,0.0,0.0,0.750,0.328,0.362,0.638
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,7c Jc Ah Jd,1.000,0.750,0.750,0.417,0.50,0.25,0.25,0.00,1.0,0.0,0.0,0.417,0.174,0.584,0.416
96,3c 3h Qh 4c,0.833,0.167,0.083,0.083,0.50,0.00,0.50,0.00,1.0,0.0,0.0,0.250,0.351,0.338,0.662
97,Th Jc Qd 3h,0.833,0.750,0.667,0.083,0.25,0.25,0.50,0.00,0.0,0.0,0.0,0.250,0.229,0.493,0.507
98,7c 8c Ks 2c,0.917,0.500,0.417,0.000,0.75,0.00,0.00,0.25,0.0,0.0,1.0,0.083,0.376,0.312,0.688
